#  LLM 성능평가 개요

### **학습 목표:** A/B 테스트를 수행하여 LLM 애플리케이션의 성능 평가를 적용한다

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [3]:
import os
from glob import glob

from pprint import pprint
import json

`(3) langfuase handler 설정`

In [4]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

`(4) Test Data`

In [5]:
# Test 데이터셋에 대한 QA 생성 결과를 리뷰한 후 다시 로드
import pandas as pd
df_qa_test = pd.read_excel("data/testset.xlsx")

print(f"테스트셋: {df_qa_test.shape[0]}개 문서")
df_qa_test.head(2)

테스트셋: 49개 문서


,user_input,reference_contexts,reference,synthesizer_name
0,"Tesla, Inc.는 미국에서 어떤 역할을 하고 있으며, 이 회사의 주요 제품과 ...","['Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회...","Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사로, 전기 자동차(...",single_hop_specifc_query_synthesizer
1,Forbes Global 2000에서 테슬라 순위 뭐야?,['Tesla의 차량 생산은 2008년 Roadster로 시작하여 Model S (...,테슬라는 Forbes Global 2000에서 69위에 랭크되었습니다.,single_hop_specifc_query_synthesizer


---

## **LLM 애플리케이션 평가**

- **AI 평가**는 데이터셋, 평가자, 평가 방법론 세 가지 핵심 요소로 구성되며, 초기에는 **10-20개의 고품질 예제**로 시작하는 것이 효과적

- 평가 방식은 **인간 평가**와 **자동화 평가** 두 트랙으로 진행되며, 주관적 판단이 필요한 초기에는 인간 평가를, 확장이 필요한 경우 휴리스틱 기반 자동화 평가를 활용

- 평가는 **오프라인**과 **온라인** 환경에서 수행되며, 벤치마킹, 테스트, 실시간 모니터링 등 상황에 맞는 방법론을 적용

- 지속적인 **CI/CD 통합**과 모니터링 시스템 구축을 통해 평가 프로세스의 효율성과 신뢰성을 확보해야 함

### 1) **벡터스토어** 로드

- **Chroma DB** 설정에서 모델, 컬렉션명, 저장 경로 지정

In [6]:
# 벡터 저장소 로드
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

In [7]:
# 벡터저장소 검색기 생성
chroma_k = chroma_db.as_retriever(
    search_kwargs={'k': 4},
)

# 벡터저장소 검색기를 사용하여 검색
query = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"

retrieved_docs = chroma_k.invoke(query)

# 검색 결과 출력
for doc in retrieved_docs:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")
    print("-"*200)
    print()

- [출처] 이 문서는 테슬라에 대한 문서입니다.
----------------------------------
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자로 인정받고 있습니다.

### Roadster (2005–2009) [출처: data\테슬라_KR.md]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

- [출처] 이 문서는 테슬라에 대한 문서입니다.
----------------------------------
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했

### 2) **BM25 검색기** 준비

- **BM25 검색기** 구현으로 문서 유사도 기반 검색 가능

- **한국어 텍스트 처리**를 위한 **Kiwi 토크나이저** 설정

- 참고: https://github.com/bab2min/kiwipiepy

In [8]:
# korean_docs 파일을 로드 (jsonlines 파일)
def load_jsonlines(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        docs = [json.loads(line) for line in f]
    return docs

korean_docs = load_jsonlines('data/korean_docs_final.jsonl')
print(f"로드된 문서: {len(korean_docs)}개")
pprint(korean_docs[0])

로드된 문서: 39개
('{"id":null,"metadata":{"source":"data/테슬라_KR.md","company":"테슬라","language":"ko"},"page_content":"<Document>\\nTesla, '
 'Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 '
 '전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc '
 'Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 '
 '2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 '
 "되었습니다.\\n</Document>\\n<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 "
 '문서입니다.</Source>","type":"Document"}')


In [9]:
from langchain.schema import Document  # Document 클래스 임포트

# 문자열 리스트를 Document 객체로 변환
if isinstance(korean_docs[0], str):  # 첫 번째 항목이 문자열인지 확인
    documents = [
        Document(
            page_content=json.loads(data)['page_content'],  # 문자열을 파이썬 객체로 변환
            metadata=json.loads(data)['metadata']
        )
        for i, data in enumerate(korean_docs)
    ]
else:
    documents = korean_docs

print(f"변환된 문서: {len(documents)}개")
pprint(documents[0])

변환된 문서: 39개
Document(metadata={'source': 'data/테슬라_KR.md', 'company': '테슬라', 'language': 'ko'}, page_content="<Document>\nTesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 되었습니다.\n</Document>\n<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 문서입니다.</Source>")


In [10]:
# BM25 검색기를 사용하기 위한 준비
from krag.tokenizers import KiwiTokenizer
from krag.retrievers import KiWiBM25RetrieverWithScore

kiwi_tokenizer = KiwiTokenizer(
    model_type='knlm',    # Kiwi 언어 모델 타입
    typos='basic'         # 기본 오타교정
    )

bm25_db = KiWiBM25RetrieverWithScore(
        documents=documents,
        kiwi_tokenizer=kiwi_tokenizer,
        k=4,
    )

In [11]:
# BM25 검색기를 사용하여 문서 검색
query = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
retrieved_docs = bm25_db.invoke(query, 2)

# 검색 결과 출력
for doc in retrieved_docs:
    print(f"BM25 점수: {doc.metadata["bm25_score"]:.2f}")
    print(f"\n{doc.page_content}\n[출처: {doc.metadata['source']}]")
    print("-"*200)

BM25 점수: 24.53

<Document>
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자로 인정받고 있습니다.

### Roadster (2005–2009)
</Document>
<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 문서입니다.</Source>
[출처: data/테슬라_KR.md]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
BM25 점수: 22.21

<Document>
Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도

### 3) **Emsemble Hybrid Search** 준비

- **BM25**, **벡터 검색** 결과를 **rank-fusion** 알고리즘으로 통합 (**EnsembleRetriever**)

- 각 검색기의 **순위 점수**를 고려한 최종 순위 결정

- **중복 문서** 제거와 **재순위화** 자동 수행

- 두 검색 방식의 **장점을 결합**해 검색 품질 향상

In [13]:
from langchain.retrievers import EnsembleRetriever

# 검색기 초기화
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_db, chroma_k],
    weights=[0.5, 0.5],
)

In [ ]:
query = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
retrieved_docs = hybrid_retriever.invoke(query)

# 검색 결과 출력
for doc in retrieved_docs:
    print(f"\n{doc.page_content}\n[출처: {doc.metadata['source']}]")
    print("-"*200)

---

## [실습] **RAG 성능 A/B 테스트**

- **LangChain 평가기**를 사용하여 RAG 답변의 품질을 평가합니다.

- 다음과 같은 **사용자 정의 평가 기준**을 정의하여 평가합니다. (예시)
    - Conciseness (간결성): 불필요한 반복이나 장황함 없이 핵심 내용 전달
    - Helpfulness (유용성): 실질적인 도움이 되는 정도
    - Harmfulness/Maliciousness (유해성): 해로운 내용 포함 여부

- 요구 사항:
    - 올라마(Ollama)에서 다운로드한 오픈소스 모델 성능을 gpt-4.1-mini 모델의 성능과 비교
    - 평가자 모델은 gpt-4.1 사용
    - 사용자 정의 프롬프트 사용
    - df_qa_test 전체 테스트셋에 대해서 평가를 수행
    - Reference-free 평가와 Reference-based 평가를 각각 수행 (1개 이상)


In [14]:
from langchain.evaluation import load_evaluator
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.runnables import RunnableConfig, RunnablePassthrough, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import List, Dict

def rag_bot(
    question: str,
    retriever: BaseRetriever,
    llm: BaseChatModel,
    config: RunnableConfig | None = None,
) -> Dict[str, str | List[Document]]:
    """
    문서 검색 기반 질의응답 수행
    """
    docs = retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)

    system_prompt = f"""문서 기반 질의응답 어시스턴트입니다.
- 제공된 문서만 참고하여 답변
- 불확실할 경우 '모르겠습니다' 라고 응답
- 3문장 이내로 답변

[문서]
{context}"""

    prompt = ChatPromptTemplate.from_messages(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "\n\n[질문]{question}\n\n[답변]\n"},
        ]
    )

    docqa_chain = {
        "context": lambda x: context,
        "question": RunnablePassthrough(),
        "docs": lambda x: docs,
    } | RunnableParallel({
        "answer": prompt | llm | StrOutputParser(),
        "documents": lambda x: x["docs"],
    })

    return docqa_chain.invoke(question, config=config)

# 모델별 비교
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

gpt_model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
gemini_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Reference-free 평가
def reference_free_evaluation():
    # 사용자 정의 평가 기준
    custom_criteria = {
        "conciseness": "불필요한 반복이나 장황함 없이 핵심 내용을 전달하는가?",
        "helpfulness": "실질적인 도움이 되는 정도는 어떠한가?",
        "harmfulness": "해로운 내용이 포함되어 있지 않은가?"
    }
    
    evaluator = load_evaluator(
        "pairwise_string",
        criteria=custom_criteria,
        llm=ChatOpenAI(model="gpt-4.1", temperature=0),
        callbacks=[langfuse_handler]
    )
    
    for idx, row in df_qa_test.iterrows():
        question = row['user_input']
        
        gpt_response = rag_bot(
            question=question,
            retriever=hybrid_retriever,
            llm=gpt_model,
            config={"callbacks": [langfuse_handler]}
        )
        
        gemini_response = rag_bot(
            question=question,
            retriever=hybrid_retriever,
            llm=gemini_model,
            config={"callbacks": [langfuse_handler]}
        )
        
        result = evaluator.evaluate_string_pairs(
            prediction=gpt_response["answer"],
            prediction_b=gemini_response["answer"],
            input=question
        )
        
        print(f"Q{idx+1}: {result['value']} 승리")

# Reference-based 평가
def reference_based_evaluation():
    # 사용자 정의 평가 기준
    custom_criteria = {
        "conciseness": "불필요한 반복이나 장황함 없이 핵심 내용을 전달하는가?",
        "helpfulness": "실질적인 도움이 되는 정도는 어떠한가?",
        "harmfulness": "해로운 내용이 포함되어 있지 않은가?"
    }
    
    evaluator = load_evaluator(
        "labeled_pairwise_string",
        criteria=custom_criteria,
        llm=ChatOpenAI(model="gpt-4.1", temperature=0),
        callbacks=[langfuse_handler]
    )
    
    for idx, row in df_qa_test.iterrows():
        question = row['user_input']
        reference = row['reference']
        
        gpt_response = rag_bot(
            question=question,
            retriever=hybrid_retriever,
            llm=gpt_model,
            config={"callbacks": [langfuse_handler]}
        )
        
        gemini_response = rag_bot(
            question=question,
            retriever=hybrid_retriever,
            llm=gemini_model,
            config={"callbacks": [langfuse_handler]}
        )
        
        result = evaluator.evaluate_string_pairs(
            prediction=gpt_response["answer"],
            prediction_b=gemini_response["answer"],
            input=question,
            reference=reference
        )
        
        print(f"Q{idx+1}: {result['value']} 승리")

print("Reference-free 평가 시작")
reference_free_evaluation()

print("Reference-based 평가 시작")  
reference_based_evaluation()

Reference-free 평가 시작
Q1: A 승리
Q2: B 승리
Q3: A 승리
Q4: A 승리
Q5: B 승리
Q6: B 승리
Q7: None 승리
Q8: A 승리
Q9: A 승리
Q10: A 승리
Q11: A 승리
Q12: A 승리
Q13: B 승리
Q14: A 승리
Q15: B 승리
Q16: None 승리
Q17: A 승리
Q18: B 승리
Q19: A 승리
Q20: A 승리
Q21: A 승리
Q22: B 승리
Q23: B 승리
Q24: None 승리
Q25: None 승리
Q26: A 승리
Q27: A 승리
Q28: B 승리
Q29: B 승리
Q30: B 승리
Q31: A 승리
Q32: A 승리
Q33: A 승리
Q34: None 승리
Q35: B 승리
Q36: A 승리
Q37: A 승리
Q38: B 승리
Q39: B 승리
Q40: B 승리
Q41: A 승리
Q42: B 승리
Q43: A 승리
Q44: B 승리
Q45: A 승리
Q46: A 승리
Q47: A 승리
Q48: B 승리
Q49: A 승리
Reference-based 평가 시작
Q1: A 승리
Q2: B 승리
Q3: B 승리
Q4: A 승리
Q5: B 승리
Q6: B 승리
Q7: B 승리
Q8: A 승리
Q9: A 승리
Q10: A 승리
Q11: B 승리
Q12: A 승리
Q13: A 승리
Q14: B 승리
Q15: A 승리
Q16: None 승리
Q17: A 승리
Q18: B 승리
Q19: B 승리
Q20: A 승리
Q21: A 승리
Q22: A 승리
Q23: None 승리
Q24: None 승리
Q25: A 승리
Q26: A 승리
Q27: A 승리
Q28: B 승리
Q29: B 승리
Q30: B 승리
Q31: A 승리
Q32: None 승리
Q33: B 승리
Q34: None 승리
Q35: A 승리
Q36: A 승리
Q37: B 승리
Q38: B 승리
Q39: A 승리
Q40: B 승리
Q41: A 승리
Q42: B 승리
Q43: A 승리
Q44: A 승리
Q45: A 승리
Q46: 